In [3]:
from fastaUtils import *

In [4]:
# Download the full Swiss-Prot database
swissprotFile = downloadDataBase(sourceURL="https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz",
                                 outputFile="swissprot.fasta")

  https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz
  This may take several minutes...
Progress: 100.0%
Download complete. Decompressing...
  The database was downloaded successfully!
  File: swissprot.fasta


In [5]:
# run an analysis on the sequences in the downloaded database
swissProtStats = analyzeFastaFile("./swissprot.fasta")
# print the statistics
printFastaStats(swissProtStats)


Analysis for: swissprot.fasta
Number of sequences:   574,627
Total length (bp/aa):  208,482,574

Length statistics:
  Average length:      362.81
  Standard deviation:  344.74
  Minimum length:      2
  Maximum length:      35213

Quartiles:
  25%: 169.00
  50% (median): 295.00
  75%: 450.00


# MMseqs2 expriments

In [1]:
from mmseqs2Utils import MMseqsUtils

In [2]:
tool = MMseqsUtils()

In [3]:
queryDB = tool.createDB("queries.fasta", "./customDatabase/query", "queryDB")
targetDB = tool.createDB("swissprot.fasta", "./customDatabase/target", "targetDB")

In [4]:
resultsPath = "./customDatabase/results.tsv"
resultsDF = tool.runSearch(queryDB, targetDB, resultsPath, threads = 16)

query,target,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,qlen,tlen,qcov,tcov
Search completed with sensitivity = 7.5


In [5]:
print(resultsDF.head(n=10).to_string(index=False))

     query target  pident  alnlen  mismatch  gapopen  qstart  qend  tstart  tend       evalue  bits  qlen  tlen  qcov  tcov
A0A0A3CWN7 Q9P838    89.1      60         6        0       1    60       1    60 4.790000e-30   114    60    60 1.000 1.000
A0A0A3CWN7 Q99380    63.2      32        11        0       1    32       1    32 3.532000e-05    43    60    36 0.533 0.889
    H1PYY5 Q8RIF7    77.1      95        22        0       1    95       1    95 1.697000e-42   152    95    95 1.000 1.000
    H1PYY5 B0TC58    49.8      92        45        0       4    95       5    95 3.195000e-20    88    95    95 0.968 0.958
    H1PYY5 Q0TMP8    47.9      92        47        0       1    92       3    93 7.596000e-19    84    95    97 0.968 0.938
    H1PYY5 Q8XHS5    47.9      92        47        0       1    92       3    93 7.596000e-19    84    95    97 0.968 0.938
    H1PYY5 Q0SQE6    47.9      92        47        0       1    92       3    93 7.596000e-19    84    95    97 0.968 0.938
    H1PY

In [ ]:
runtimes = []
numOfThreads = [1, 2, 4, 8, 16, 32]

for numThreads in numOfThreads:
    resultFileName = f"./customDatabase/results_{numThreads}threads.tsv"
    t0 = time.time()
    results = runMmseqsSearch(queryDB, targetDB, resultFileName, threads = numThreads)
    t1 = time.time()
    runtimes.append(t1-t0)
    print(f"Number of threads used: {numThreads}.")

print(runtimes)
np.savetxt('./customDatabase/runTime_threads.csv', runtimes, delimiter=',')

# Diamond experiments

In [1]:
from diamondUtils import *

In [4]:
queryDB = "./queries.fasta"
targetDB = createDiamondDataBase("swissprot.fasta", "./customDatabase/target_diamond", "targetDB")

Preparing database...


diamond v2.2.1.181 (C) Max Planck Society for the Advancement of Science, Benjamin J. Buchfink, University of Tuebingen
Documentation, support and updates available at http://www.diamondsearch.org
Please cite: http://dx.doi.org/10.1038/s41592-021-01101-x Nature Methods (2021)

#CPU threads: 2
Scoring parameters: (Matrix=BLOSUM62 Lambda=0.267 K=0.041 Penalties=11/1)
Database input file: swissprot.fasta
Opening the database file...  [0.009s]
Loading sequences...  [0.548s]
Masking sequences...  [2.672s]
Writing sequences...  [0.123s]
Hashing sequences...  [0.045s]
Loading sequences...  [0.001s]
Writing trailer...  [0.004s]
Closing the input file...  [0s]
Closing the database file...  [0.001s]

Database sequences  574627
  Database letters  208482574
     Database hash  145a58bd06146ceb2e43d59e0638c261
        Total time  3.407000s


In [5]:
results = runDiamondSearch(queryDB, targetDB, "./customDatabase/results_diamond.tsv", threads = 16, sensitivity='ultra-sensitive')

In [ ]:
runtimes = []
numOfThreads = [1, 2, 4, 8, 16, 32]

for numThreads in numOfThreads:
    resultFileName = f"./customDatabase/results_{numThreads}threads.tsv"
    t0 = time.time()
    results = runDiamondSearch(queryDB, targetDB, resultFileName, sensitivity='ultra-sensitive', threads = numThreads)
    t1 = time.time()
    runtimes.append(t1-t0)
    print(f"Number of threads used: {numThreads}.")

print(runtimes)
np.savetxt('./customDatabase/runTime_threads.csv', runtimes, delimiter=',')